# Obligatorio Taller de Tecnologías 2: Agentic RAG

Universidad ORT Uruguay, 2026.

Este notebook integra y demuestra los módulos de `tt2_agentic_rag`. La lógica vive en `src/`; acá solo se usa.
La versión autocontenida para Google Colab es `notebooks/obligatorio_colab.ipynb`.

Requisitos: `pip install -e .` y un `.env` con `GROQ_API_KEY` y `PINECONE_API_KEY` (ver `.env.example`).

## 1. Configuración

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

import tt2_agentic_rag

# Raíz del repo, tanto si el kernel arranca en notebooks/ como en la raíz
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
load_dotenv(ROOT / ".env")

PAPERS_DIR = ROOT / (os.getenv("PAPERS_DIR") or "data/papers")
DATASET_PATH = ROOT / (os.getenv("DATASET_PATH") or "data/datasets/netflix_titles.csv")
MEMORY_DB_PATH = ROOT / (os.getenv("MEMORY_DB_PATH") or "storage/memory.sqlite")
LLM_MODEL = (os.getenv("LLM_MODEL") or "qwen/qwen3.8-27b")

for key in ("GROQ_API_KEY", "PINECONE_API_KEY"):
    if not os.getenv(key):
        raise ValueError(f"Falta {key} en .env (ver .env.example).")

print("tt2_agentic_rag", tt2_agentic_rag.__version__)
print("Papers: ", PAPERS_DIR)
print("Dataset:", DATASET_PATH)

## 2. Ingesta de papers

PDFs → páginas (`PyPDFLoader`) → chunks de 800 caracteres con solapamiento de 120 (`RecursiveCharacterTextSplitter`)
→ normalización de espacios → embeddings `all-MiniLM-L6-v2` → índice `research-papers` en Pinecone.

Cada chunk guarda `file_name` y `page` para citar la fuente. Si el índice ya tiene vectores no se reindexa
(`force=True` reindexa con IDs deterministas, sin duplicar).

In [ ]:
from tt2_agentic_rag import ingest

raw_papers_docs = ingest.load_pdfs(PAPERS_DIR)
paper_chunks = ingest.split_documents(raw_papers_docs)

print(f"\nPáginas originales: {len(raw_papers_docs)}")
print(f"Chunks generados y limpios: {len(paper_chunks)}")
sample = paper_chunks[0]
print(f"\nMuestra (Paper: {sample.metadata['file_name']}, Pág: {sample.metadata['page']}):")
print(sample.page_content[:200] + "...")

In [ ]:
vectorstore_papers = ingest.build_index(PAPERS_DIR)
print("Vectores en 'research-papers':", ingest.index_vector_count(ingest.PAPERS_INDEX_NAME))

## 3. Recuperación semántica (RAG)

In [ ]:
from tt2_agentic_rag import rag

for doc in rag.search("attention mechanism transformer architecture", k=3):
    print(f"[{doc.metadata['file_name']} | pág. {ingest.as_int(doc.metadata['page'])}] {doc.page_content[:150]}...")

In [ ]:
# La misma búsqueda, a través de la tool que usa el agente
print(rag.query_research_papers.invoke("scaling laws compute data model size")[:1200])

## 4. Dataset de Kaggle: Netflix Titles

Limpieza: nulos de texto → cadena vacía y colapso de espacios, sin tocar el vocabulario. Título, tipo, géneros,
director, elenco y descripción se concatenan con etiquetas (`Title: ...`) en `text_for_embedding`; país, año y tipo
quedan como metadatos filtrables. Índice `netflix-catalog` en Pinecone, con `show_id` como ID.

In [ ]:
from tt2_agentic_rag import dataset

df_netflix = dataset.load_dataset(DATASET_PATH)
print(f"Filas: {df_netflix.shape[0]} | Columnas: {df_netflix.shape[1]}")

df_netflix = dataset.prepare_dataset(df_netflix)
print("\nMuestra de texto preparado para embeddings:")
print(df_netflix["text_for_embedding"].iloc[0])
df_netflix[["title", "type", "country", "release_year"]].head(3)

In [ ]:
vectorstore_netflix = dataset.build_index(df_netflix)
print("Vectores en 'netflix-catalog':", ingest.index_vector_count(dataset.NETFLIX_INDEX_NAME))

In [ ]:
def mostrar(docs):
    for i, doc in enumerate(docs, start=1):
        m = doc.metadata
        print(f"  {i}. {m.get('title')} ({m.get('type')}) | {m.get('country')} | {ingest.as_int(m.get('release_year'))}")


print("TEST 1: búsqueda semántica (trama)")
mostrar(dataset.search("detective solving murder mystery in a small dark town", k=3))

print("\nTEST 2: búsqueda conceptual (ciencia ficción)")
mostrar(dataset.search("space travel and exploration of alien civilizations", k=3))

print("\nTEST 3: filtro por metadatos (películas de España)")
mostrar(dataset.search(
    "crime and thrilling heist", k=3,
    filter={"country": {"$eq": "Spain"}, "type": {"$eq": "Movie"}},
))

## 5. Agente LangGraph

Grafo armado a mano en `agent.build_agent` (`START → llm → tools → llm … → END`, sin `create_agent`,
`create_react_agent`, `ToolNode` ni `tools_condition`). El LLM decide con tool calling entre:

- `query_research_papers` (papers),
- `query_netflix_catalog` (dataset),
- `get_user_memory` (memoria a largo plazo del usuario).

In [ ]:
from langchain_groq import ChatGroq

from tt2_agentic_rag.agent import build_agent
from tt2_agentic_rag.dataset import query_netflix_catalog
from tt2_agentic_rag.memory import LongTermMemory, get_checkpointer, get_store
from tt2_agentic_rag.rag import query_research_papers

llm = ChatGroq(model=LLM_MODEL, temperature=0, max_tokens=500)

checkpointer = get_checkpointer(MEMORY_DB_PATH)  # memoria a corto plazo (por thread_id)
store = get_store(MEMORY_DB_PATH)
long_term_memory = LongTermMemory(llm)           # hechos del usuario (por user_id)

agent = build_agent(
    llm=llm,
    tools=[query_netflix_catalog, query_research_papers, long_term_memory.as_tool()],
    checkpointer=checkpointer,
    store=store,
)
print("¡Agente compilado!")

In [ ]:
# Estructura del grafo (Mermaid)
print(agent.get_graph().draw_mermaid())

## 6. Memoria y conversaciones múltiples

- **Corto plazo**: el checkpointer SQLite guarda el historial de cada `thread_id`; sobrevive a reinicios del kernel.
- **Largo plazo**: al cierre de cada turno, `LongTermMemory.save_from_turn` extrae hechos del usuario con el LLM,
  descarta los que quedan contradichos y guarda el nuevo en Pinecone (`user-memory`, filtrado por `user_id`).
  En otra conversación el agente los recupera con `get_user_memory`.

Cada ejecución del notebook usa IDs de hilo nuevos para que la demo arranque de cero.

In [ ]:
import uuid

from langchain_core.messages import AIMessage, HumanMessage

RUN = uuid.uuid4().hex[:6]


def conversar(mensaje: str, thread_id: str, user_id: str = "nicolas") -> None:
    config = {"configurable": {"thread_id": f"{thread_id}-{RUN}", "user_id": user_id}}
    print(f"\n🧑 [{user_id} | {thread_id}]: {mensaje}")

    previos = len(agent.get_state(config).values.get("messages", []))
    response = agent.invoke({"messages": [HumanMessage(content=mensaje)]}, config=config)
    nuevos = response["messages"][previos:]

    tools = [c["name"] for m in nuevos if isinstance(m, AIMessage) for c in m.tool_calls]
    print(f"🔧 Herramientas: {', '.join(tools) or 'ninguna'}")
    bot_reply = response["messages"][-1]
    print(f"🤖 Asistente: {bot_reply.content}")

    # Memoria a largo plazo al cierre del turno
    long_term_memory.save_from_turn([HumanMessage(content=mensaje), bot_reply], user_id=user_id)
    print("-" * 75)

## 7. Demostración y conclusiones

In [ ]:
# 1. Small-talk (no debe disparar herramientas)
conversar("¡Hola! ¿Cómo andás? Me llamo Nicolás y me gustan mucho los thrillers psicológicos.", thread_id="hilo-1")

In [ ]:
# 2. Consulta a los papers (RAG)
conversar("¿Qué explican los documentos sobre la arquitectura Transformer y el encoder?", thread_id="hilo-1")

In [ ]:
# 3. Memoria a corto plazo (referencia implícita al turno anterior en el mismo hilo)
conversar("¿En qué página del paper estaba eso que me acabás de resumir?", thread_id="hilo-1")

In [ ]:
# 4. Consulta al dataset de Netflix (RAG)
conversar("Buscame películas o series en Netflix relacionadas con detectives e investigación criminal.", thread_id="hilo-1")

In [ ]:
# 5. Memoria a largo plazo: hilo nuevo, mismo user_id
conversar("Hola de nuevo, arrancamos un chat nuevo. ¿Te acordás cómo me llamo y qué género de películas me gusta?", thread_id="hilo-2")

In [ ]:
# 6. Control de alucinaciones (información inexistente en las fuentes)
conversar("¿Qué dice el paper de Attention Is All You Need sobre el cultivo de yerba mate en Uruguay?", thread_id="hilo-2")

In [ ]:
# Las conversaciones son independientes: hilo-1 tiene su propio historial
for hilo in ("hilo-1", "hilo-2"):
    state = agent.get_state({"configurable": {"thread_id": f"{hilo}-{RUN}"}})
    print(f"{hilo}: {len(state.values['messages'])} mensajes guardados en SQLite")

## 8. Interfaz web con Gradio (opcional)

Levanta una interfaz web interactiva reutilizando el `agent` y la `long_term_memory` ya inicializados.

In [ ]:
from tt2_agentic_rag.ui import create_demo

demo = create_demo(
    agent,
    save_memory_fn=lambda msgs, uid: long_term_memory.save_from_turn(msgs, user_id=uid),
)
demo.launch()